In [ ]:
from time import sleep
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from rich import print


#1. 状態を宣言
class OverAllState(TypedDict):
    username: str
    age: int


#2. ノードを宣言
def node_a(state: OverAllState) -> OverAllState:
    username = interrupt("お名前を入力してください:")
    return {
        "username": username
    }


def node_b(state: OverAllState) -> OverAllState:
    sleep(1)
    age = interrupt("年齢を入力してください:")
    return {
        "age": age
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)

builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

#4. チェックポインターバックエンドを追加
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

from IPython.display import display

display(graph)

#5. 最初のグラフを実行 => 中断をトリガー
config = {"configurable": {"thread_id": "123"}}
interrupt_res = graph.invoke({}, config=config)
print(interrupt_res)

In [ ]:
# 6. 実行を再開
resume_map = {}
for i in interrupt_res['__interrupt__']:
    user_input = input(f"{i.value}:")
    if "年齢" in i.value:
        resume_map[i.id] = int(user_input)
    else:
        resume_map[i.id] = user_input

resumed_res = graph.invoke(Command(resume=resume_map), config=config)
print(resumed_res)
